In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

In [2]:
ROOT = Path.cwd().parent

DATA_DIR = ROOT / "data" / "processed"
RAW_DIR = ROOT / "data" / "raw"
CHARTS_DIR = ROOT / "charts"

CHARTS_DIR.mkdir(exist_ok=True)

In [3]:
nav = pd.read_csv(DATA_DIR / "clean_nav.csv")
performance = pd.read_csv(DATA_DIR / "clean_performance.csv")
benchmark = pd.read_csv(RAW_DIR / "10_benchmark_indices.csv")

In [4]:
print(performance.columns.tolist())
print(benchmark.columns.tolist())

['amfi_code', 'scheme_name', 'fund_house', 'category', 'plan', 'return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct', 'benchmark_3yr_pct', 'alpha', 'beta', 'sharpe_ratio', 'sortino_ratio', 'std_dev_ann_pct', 'max_drawdown_pct', 'aum_crore', 'expense_ratio_pct', 'morningstar_rating', 'risk_grade', 'anomaly_flag']
['date', 'index_name', 'close_value']


In [5]:
performance[
[
    "return_1yr_pct",
    "return_3yr_pct",
    "return_5yr_pct",
    "alpha",
    "beta",
    "sharpe_ratio",
    "sortino_ratio",
    "max_drawdown_pct",
    "expense_ratio_pct"
]
].head()

,return_1yr_pct,return_3yr_pct,return_5yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,max_drawdown_pct,expense_ratio_pct
0,12.42,12.36,14.45,0.87,0.89,0.88,1.29,-21.70,1.54
1,15.25,11.30,14.23,1.78,0.87,0.81,1.29,-24.43,0.66
2,24.56,23.39,20.67,1.23,0.89,0.94,1.35,-13.35,1.43
3,20.59,23.14,21.82,1.13,1.04,0.93,1.67,-24.78,0.72
4,5.34,6.07,5.43,1.60,0.22,1.52,2.11,-2.30,0.77


In [6]:
performance[
[
    "return_1yr_pct",
    "return_3yr_pct",
    "return_5yr_pct",
    "alpha",
    "beta",
    "sharpe_ratio",
    "sortino_ratio",
    "max_drawdown_pct"
]
].isna().sum()

return_1yr_pct      0
return_3yr_pct      0
return_5yr_pct      0
alpha               0
beta                0
sharpe_ratio        0
sortino_ratio       0
max_drawdown_pct    0
dtype: int64

# 1. Daily Return Analysis

## Objective

To calculate the daily percentage return for each mutual fund scheme using historical NAV data. Daily returns provide the foundation for measuring risk, volatility, and performance metrics such as Sharpe Ratio, Sortino Ratio, Alpha, and Beta.

In [7]:
nav.head()

,amfi_code,date,nav
0,100016,2022-01-03,520.4608
1,100016,2022-01-04,515.0971
2,100016,2022-01-05,521.7239
3,100016,2022-01-06,515.7880
4,100016,2022-01-07,515.1639


In [8]:
nav.info()

<class 'pandas.DataFrame'>
RangeIndex: 46000 entries, 0 to 45999
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   amfi_code  46000 non-null  int64  
 1   date       46000 non-null  str    
 2   nav        46000 non-null  float64
dtypes: float64(1), int64(1), str(1)
memory usage: 1.1 MB


In [9]:
nav.describe()

,amfi_code,nav
count,46000.000000,46000.000000
mean,120247.000000,269.570265
std,14352.317221,577.187060
min,100016.000000,26.136600
25%,118632.750000,69.170425
50%,119551.500000,122.732150
75%,120842.250000,260.338675
max,149324.000000,4268.549700


In [10]:
nav["date"] = pd.to_datetime(nav["date"])

In [11]:
nav = nav.sort_values(["amfi_code", "date"])

In [13]:
nav["daily_return"] = (
    nav.groupby("amfi_code")["nav"]
       .pct_change()
)

In [14]:
nav.head(10)

,amfi_code,date,nav,daily_return
0,100016,2022-01-03,520.4608,NaN
1,100016,2022-01-04,515.0971,-0.010306
2,100016,2022-01-05,521.7239,0.012865
3,100016,2022-01-06,515.7880,-0.011377
4,100016,2022-01-07,515.1639,-0.001210
5,100016,2022-01-10,510.7136,-0.008639
6,100016,2022-01-11,513.5542,0.005562
7,100016,2022-01-12,512.3195,-0.002404
8,100016,2022-01-13,510.2445,-0.004050
9,100016,2022-01-14,514.3636,0.008073


In [18]:
nav["daily_return"].describe()


count    45960.000000
mean         0.000631
std          0.010290
min         -0.058102
25%         -0.005042
50%          0.000340
75%          0.006324
max          0.064713
Name: daily_return, dtype: float64

In [22]:
nav["daily_return"].isna().sum()

np.int64(40)

In [23]:
nav.head(10)

,amfi_code,date,nav,daily_return
0,100016,2022-01-03,520.4608,NaN
1,100016,2022-01-04,515.0971,-0.010306
2,100016,2022-01-05,521.7239,0.012865
3,100016,2022-01-06,515.7880,-0.011377
4,100016,2022-01-07,515.1639,-0.001210
5,100016,2022-01-10,510.7136,-0.008639
6,100016,2022-01-11,513.5542,0.005562
7,100016,2022-01-12,512.3195,-0.002404
8,100016,2022-01-13,510.2445,-0.004050
9,100016,2022-01-14,514.3636,0.008073
